# Imports

In [1]:

import scipy.io as sio
import numpy as np
import pandas as pd
from pathlib import Path
import re

# Goals
1. Learn fundamentals of feature selection - for classification task.
2. Replicate the pipeline for anomaly detection. 
   1. Pre-computed feature based models make it easier reason about the model results - more interpretability - more trust from operators/engineers in model results

# Feature Selection Pipeline for Classification task
- exactly replicate catch22 paper but with tsfresh features
- Step 1: exactly replace catch22 paper's results
- Step 2: re-org code
- Step 3: apply the pipeline to tsfresh paper

## Data Load: UCR Clasification data

### Mat file structure (from `op_importance` repo)

Each `HCTSA_{dataset_name}_N.mat` contains:
- **`TS_DataMat`**: feature matrix, shape `(n_timeseries, n_operations)` — rows are time series samples, columns are hctsa features
- **`TimeSeries`**: struct array with `filename`, `keywords` (labels embedded as first CSV token), `n_samples`
- **`Operations`**: struct array with `code_string`, `name`, `keywords`, `id`, `master_id`
- **`MasterOperations`**: master operation metadata

Source: [figshare — precomputed HCTSA matrices for UCR 2018](https://figshare.com/articles/dataset/Computed_HCTSA_matrices_for_the_UEA_UCR_2018_time-series_classification_tasks/6865163)

In [34]:
def load_hctsa_mat(mat_file_path):
    """Load a precomputed HCTSA .mat file and return feature matrix, labels, and operation metadata.
    
    Returns:
        data: np.ndarray, shape (n_timeseries, n_operations), special values replaced with NaN
        labels: np.array of class labels (strings)
        operations: dict with keys 'id', 'name', 'code_string', 'keywords', 'master_id'
    """
    mat = sio.loadmat(mat_file_path)
    
    # Feature matrix
    data = mat['TS_DataMat'].astype(np.float64)
    
    # TS_Quality tracks whether each feature computation succeeded for each time series.
    # Source: hctsa/Calculation/TS_CalculateFeatureVector.m
    # Quality codes:
    #   0 = good (real-valued, finite output)
    #   1 = fatal error during computation
    #   2 = NaN output
    #   3 = +Inf output
    #   4 = -Inf output
    #   5 = complex number output
    # HCTSA zeros out TS_DataMat at special positions instead of storing NaN/Inf,
    # so we use TS_Quality to restore them as NaN for proper downstream handling.
    quality = mat['TS_Quality']
    data[quality != 0] = np.nan
    
    # Extract labels from TimeSeries keywords (first comma-separated token)
    ts_struct = mat['TimeSeries']
    n_ts = ts_struct.shape[0]
    labels = np.array([str(ts_struct[i, 0]['Keywords'][0]).split(',')[0] for i in range(n_ts)])
    
    # Rename string "nan" labels (e.g. AALTDChallenge) to "unknown"
    labels = np.where(labels == 'nan', 'unknown', labels)
    
    # Extract operation (feature) metadata
    op_struct = mat['Operations']
    n_ops = op_struct.shape[0]
    operations = {
        'code_string': [str(op_struct[i, 0]['CodeString'][0]) for i in range(n_ops)],
        'name':        [str(op_struct[i, 0]['Name'][0]) for i in range(n_ops)],
        'keywords':    [str(op_struct[i, 0]['Keywords'][0]) for i in range(n_ops)],
        'id':          [int(op_struct[i, 0]['ID'][0][0]) for i in range(n_ops)],
        'master_id':   [int(op_struct[i, 0]['MasterID'][0][0]) for i in range(n_ops)],
    }
    
    return data, labels, operations
    
def load_all_tasks(mat_dir, task_names: list=None):
    """Load all HCTSA .mat files from a directory.
    
    Returns:
        all_tasks: pd.DataFrame with features, labels, dataset_name
        hctsa_ops_map: pd.DataFrame mapping feature names to metadata
    """
    mat_dir = Path(mat_dir)
    all_tasks = []
    for mat_path in sorted(mat_dir.glob('HCTSA_*.mat')):
        dataset_name = mat_path.stem.replace('HCTSA_', '')
        if dataset_name not in task_names:
            continue
        data, labels, ops_map = load_hctsa_mat(mat_path)
        df_features = pd.DataFrame(data, columns=ops_map['name'])
        df_features['label'] = labels
        df_features['dataset_name'] = dataset_name 
        all_tasks.append(df_features)
    print(f"\nLoaded {len(all_tasks)} datasets")

    all_tasks = pd.concat(all_tasks)
    # All tasks mat file have the same ops_map
    # ops_map maps the features to keywords, ids 
    hctsa_ops_map = pd.DataFrame.from_dict(ops_map)
    return all_tasks,  hctsa_ops_map

In [ ]:

# Download .mat files from figshare link above into this directory
# Expected naming: HCTSA_{dataset_name}_N.mat
task_names_UCR_2018 = ["AALTDChallenge", "Adiac", "ArrowHead", "Beef", "BeetleFly", "BirdChicken", "CBF", "Car",
                      "ChlorineConcentration", "CinCECGtorso", "Coffee", "Computers", "CricketX", "CricketY", "CricketZ",
                      "DiatomSizeReduction", "DistalPhalanxOutlineAgeGroup", "DistalPhalanxOutlineCorrect",
                      "DistalPhalanxTW", "ECG200", "ECG5000", "ECGFiveDays", "ECGMeditation", "Earthquakes",
                      "ElectricDeviceOn", "ElectricDevices", "EpilepsyX", "EthanolLevel", "FaceAll", "FaceFour", "FacesUCR",
                      "FiftyWords", "Fish", "FordA", "FordB", "GunPoint", "Ham", "HandOutlines", "Haptics", "HeartbeatBIDMC",
                      "Herring", "InlineSkate", "InsectWingbeatSound", "ItalyPowerDemand", "LargeKitchenAppliances",
                      "Lightning2", "Lightning7", "Mallat", "Meat", "MedicalImages", "MiddlePhalanxOutlineAgeGroup",
                      "MiddlePhalanxOutlineCorrect", "MiddlePhalanxTW", "MoteStrain", "NonInvasiveFatalECGThorax1",
                      "NonInvasiveFatalECGThorax2", "NonInvasiveFetalECGThorax1", "NonInvasiveFetalECGThorax2", "OSULeaf",
                      "OliveOil", "PhalangesOutlinesCorrect", "Phoneme", "Plane", "ProximalPhalanxOutlineAgeGroup",
                      "ProximalPhalanxOutlineCorrect", "ProximalPhalanxTW", "RefrigerationDevices", "ScreenType",
                      "ShapeletSim", "ShapesAll", "SmallKitchenAppliances", "SonyAIBORobotSurface1",
                      "SonyAIBORobotSurface2", "StarLightCurves", "Strawberry", "SwedishLeaf", "Symbols",
                      "SyntheticControl", "ToeSegmentation1", "ToeSegmentation2", "Trace", "TwoLeadECG", "TwoPatterns",
                      "UWaveGestureLibraryAll", "UWaveGestureLibraryX", "UWaveGestureLibraryY", "UWaveGestureLibraryZ",
                      "Wafer", "Wine", "WordSynonyms", "Worms", "WormsTwoClass", "Yoga"]
mat_dir = Path('../datasets/ucr_hctsa_features')
all_tasks, hctsa_ops_map = load_all_tasks(mat_dir, task_names = task_names_UCR_2018)

In [ ]:
all_tasks.to_parquet(Path("../datasets/ucr_hctsa.parquet"))

## Pre-Filtering before feature selection
- remove raw data based features
- remove features which give invalid values for atleast one time series in 80% of the tasks

In [ ]:
all_tasks.info()

In [ ]:
def remove_raw_data_features(all_tasks: list[pd.DataFrame], hctsa_ops_map: pd.DataFrame) -> list[pd.DataFrame]:
    ops_tokeep = hctsa_ops_map.loc[~hctsa_ops_map["keywords"].str.contains("raw"), "name"].values
    filtered_tasks = all_tasks[[col for col in all_tasks.columns if col in ops_tokeep] + ["label", "dataset_name"]]
    return filtered_tasks

all_tasks = remove_raw_data_features(all_tasks, hctsa_ops_map)
all_tasks.shape

In [ ]:
all_tasks.to_parquet(Path("../datasets/ucr_hctsa_raw_removed.parquet"))

## Feature Normalization


In [2]:
all_tasks = pd.read_parquet(Path("../datasets/ucr_hctsa_raw_removed.parquet"))

In [ ]:
# Linear rescaling to unit interval [0, 1] per dataset, per feature
# As per paper: "For each dataset, each feature was linearly rescaled to the unit interval"
# Features with zero range (constant across all time series in a dataset) become NaN
feature_cols = [c for c in all_tasks.columns if c not in ('label', 'dataset_name')]

for i, (name, idx) in enumerate(all_tasks.groupby('dataset_name').groups.items()):
    print(name, i)
    block = all_tasks.loc[idx, feature_cols]
    min_vals = block.min()
    range_vals = block.max() - min_vals
    all_tasks.loc[idx, feature_cols] = (block - min_vals) / range_vals

print(f"Shape: {all_tasks.shape}")
# print(f"NaN count after normalization: {all_tasks[feature_cols].isna().sum().sum():,}")

In [7]:
# Linear rescaling to unit interval [0, 1] per dataset, per feature
# As per paper: "For each dataset, each feature was linearly rescaled to the unit interval"
# Features with zero range (constant across all time series in a dataset) become NaN
feature_cols = [c for c in all_tasks.columns if c not in ('label', 'dataset_name')]

def normalize_features(group):
    feat = group[feature_cols]
    min_vals = feat.min()
    range_vals = feat.max() - min_vals
    result = group.copy()
    result[feature_cols] = (feat - min_vals) / range_vals
    result['dataset_name'] = group.name
    return result

all_tasks = all_tasks.groupby('dataset_name', group_keys=False).apply(normalize_features)
print(f"Shape: {all_tasks.shape}")

/var/folders/nx/j3jrdkw97q1gxc0nkyjz0sg80000gn/T/ipykernel_81395/990264879.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result['dataset_name'] = group.name
/var/folders/nx/j3jrdkw97q1gxc0nkyjz0sg80000gn/T/ipykernel_81395/990264879.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result['dataset_name'] = group.name
/var/folders/nx/j3jrdkw97q1gxc0nkyjz0sg80000gn/T/ipykernel_81395/990264879.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which 

Shape: (147198, 6894)


In [18]:
# Count how many datasets have at least one invalid value per feature column
# Each dataset contributes at most 1 to the count per column
feature_cols = [c for c in all_tasks.columns if c not in ('label', 'dataset_name')]

def has_invalid(col):
    return col.isna().any() | np.isinf(col).any()

invalid_dataset_count = (
    all_tasks.groupby('dataset_name')[feature_cols]
    .apply(lambda grp: grp.apply(has_invalid))
    .sum()
)
# invalid_dataset_count[invalid_dataset_count>0.8*len(task_names_UCR_2018)]

In [19]:
num_datasets = all_tasks["dataset_name"].nunique()
invalid_dataset_count[invalid_dataset_count>=0.8*num_datasets]

IN_AutoMutualInfoStats_40_gaussian_modeperiodmax     84
IN_AutoMutualInfoStats_40_gaussian_pmodeperiodmax    84
IN_AutoMutualInfoStats_40_gaussian_modeperiodmin     80
IN_AutoMutualInfoStats_40_gaussian_pmodeperiodmin    80
StatAvl500                                           82
                                                     ..
MF_compare_GARCH_ar_1_3_1_3_bestqAIC                 84
MF_compare_GARCH_ar_1_3_1_3_Ks_vary_p                82
MF_compare_GARCH_ar_1_3_1_3_Ks_vary_q                82
CP_ML_StepDetect_l1pwc_10_s                          93
CP_l1pwc_sweep_lambda_0_005_095_nsegsu001            89
Length: 681, dtype: int64

In [10]:
all_tasks.columns

Index(['DN_HistogramMode_5', 'DN_HistogramMode_10', 'DN_HistogramMode_20',
       'maximum', 'miminmum', 'DN_Moments_3', 'DN_Moments_4', 'DN_Moments_5',
       'DN_Moments_6', 'DN_Moments_7',
       ...
       'SD_SurrogateTest_RP_99_o3_f', 'SD_SurrogateTest_RP_99_o3_mediqr',
       'SD_SurrogateTest_RP_99_o3_prank', 'SD_SurrogateTest_RP_99_tc3_p',
       'SD_SurrogateTest_RP_99_tc3_zscore', 'SD_SurrogateTest_RP_99_tc3_f',
       'SD_SurrogateTest_RP_99_tc3_mediqr', 'SD_SurrogateTest_RP_99_tc3_prank',
       'label', 'dataset_name'],
      dtype='str', length=6894)

In [20]:
survivors = invalid_dataset_count[invalid_dataset_count < 0.8 * num_datasets]

# Quality: % of datasets where the feature is fully valid
quality = 1 - survivors / num_datasets
print(f"Median: {quality.median():.1%}")
print(f"Worst:  {quality.min():.1%}")
print(f"≥99%:   {(quality >= 0.99).mean():.1%} of survivors")

Median: 97.8%
Worst:  20.4%
≥99%:   32.4% of survivors


In [28]:
catch22 = [
    'DN_HistogramMode_5', 'DN_HistogramMode_10',
    'SB_BinaryStats_mean_longstretch1',
    'DN_OutlierInclude_p_001_mdrmd', 'DN_OutlierInclude_n_001_mdrmd',
    'CO_f1ecac', 'CO_FirstMin_ac',
    'SP_Summaries_welch_rect_area_5_1', 'SP_Summaries_welch_rect_centroid',
    'FC_LocalSimple_mean3_stderr',
    'CO_trev_1_num', 'CO_HistogramAMI_even_2_5',
    'IN_AutoMutualInfoStats_40_gaussian_fmmi',
    'MD_hrv_classic_pnn40',
    'SB_BinaryStats_diff_longstretch0', 'SB_MotifThree_quantile_hh',
    'FC_LocalSimple_mean1_tauresrat',
    'CO_Embed2_Dist_tau_d_expfit_meandiff',
    'SC_FluctAnal_2_dfa_50_1_2_logi_prop_r1',
    'SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1',
    'SB_TransitionMatrix_3ac_sumdiagcov',
    'PD_PeriodicityWang_th0.01',
]

in_survivors = [f for f in catch22 if f in survivors.index]
missing = [f for f in catch22 if f not in survivors.index]
not_in_features = [f for f in catch22 if f not in feature_cols]

print(f"In survivors: {len(in_survivors)}/22")
print(f"Missing from survivors: {missing}")
print(f"Not in feature set at all: {not_in_features}")

In survivors: 20/22
Missing from survivors: ['CO_f1ecac', 'CO_FirstMin_ac']
Not in feature set at all: ['CO_f1ecac', 'CO_FirstMin_ac']


In [25]:
for name in ['CO_f1ecac', 'CO_FirstMin_ac', 'PD_PeriodicityWang']:
    matches = [c for c in feature_cols if name.split('_')[1] in c.lower() or name[:8] in c]
    print(f"{name}: {matches[:5]}")

CO_f1ecac: []
CO_FirstMin_ac: []
PD_PeriodicityWang: ['PD_PeriodicityWang_th0', 'PD_PeriodicityWang_th0.01', 'PD_PeriodicityWang_th0.1', 'PD_PeriodicityWang_th0.2', 'PD_PeriodicityWang_th1_sqrtN']


In [32]:
feature_cols = all_tasks.columns
for partial in ['ecac', 'FirstMin_ac']:
    matches = [c for c in feature_cols if partial in c]
    print(f"{partial}: {matches}")

ecac: []
FirstMin_ac: []


In [33]:
hctsa_ops_map

NameError: name 'hctsa_ops_map' is not defined

In [22]:
survivors.keys()

Index(['DN_HistogramMode_5', 'DN_HistogramMode_10', 'DN_HistogramMode_20',
       'maximum', 'miminmum', 'DN_Moments_3', 'DN_Moments_4', 'DN_Moments_5',
       'DN_Moments_6', 'DN_Moments_7',
       ...
       'SD_SurrogateTest_RP_99_o3_p', 'SD_SurrogateTest_RP_99_o3_zscore',
       'SD_SurrogateTest_RP_99_o3_f', 'SD_SurrogateTest_RP_99_o3_mediqr',
       'SD_SurrogateTest_RP_99_o3_prank', 'SD_SurrogateTest_RP_99_tc3_p',
       'SD_SurrogateTest_RP_99_tc3_zscore', 'SD_SurrogateTest_RP_99_tc3_f',
       'SD_SurrogateTest_RP_99_tc3_mediqr',
       'SD_SurrogateTest_RP_99_tc3_prank'],
      dtype='str', length=6211)

In [ ]:
catch22_features = 

In [13]:
quality

DN_HistogramMode_5                   1.000000
DN_HistogramMode_10                  1.000000
DN_HistogramMode_20                  1.000000
maximum                              1.000000
miminmum                             1.000000
                                       ...   
SD_SurrogateTest_RP_99_tc3_p         0.795699
SD_SurrogateTest_RP_99_tc3_zscore    0.795699
SD_SurrogateTest_RP_99_tc3_f         0.784946
SD_SurrogateTest_RP_99_tc3_mediqr    0.795699
SD_SurrogateTest_RP_99_tc3_prank     0.763441
Length: 6211, dtype: float64

In [16]:
all_tasks_un = pd.read_parquet(Path("../datasets/ucr_hctsa_raw_removed.parquet"))
feature_cols = [c for c in all_tasks.columns if c not in ('label', 'dataset_name')]


def has_invalid(col):
    return col.isna().any() | np.isinf(col).any()

invalid_dataset_count = (
    all_tasks.groupby('dataset_name')[feature_cols]
    .apply(lambda grp: grp.apply(has_invalid))
    .sum()
)
survivors = invalid_dataset_count[invalid_dataset_count < 0.8 * num_datasets]
# Per feature: fraction of individual time series that are valid, across all datasets
valid_ts_rate = all_tasks_un[feature_cols].notna().mean()

# Of survivors, what % succeed on ≥99% of time series?
(valid_ts_rate[survivors.index] >= 0.99).mean()

np.float64(0.7694413137981001)

In [ ]:
all_tasks.shape

In [ ]:
all_tasks[all_tasks["dataset_name"] == "ItalyPowerDemand"]

In [ ]:
# Debug: inspect raw .mat file structure for special value indicators
sample_mat = sio.loadmat('../datasets/ucr_hctsa_features/HCTSA_ItalyPowerDemand.mat')
print("Keys in .mat file:", [k for k in sample_mat.keys() if not k.startswith('__')])

# Check for TS_Quality matrix (HCTSA stores special-value flags here)
if 'TS_Quality' in sample_mat:
    quality = sample_mat['TS_Quality']
    print(f"\nTS_Quality shape: {quality.shape}")
    print(f"Unique quality codes: {np.unique(quality)}")
    print(f"Non-zero (special) entries: {(quality != 0).sum()} / {quality.size}")
    print(f"Columns with any special: {(quality != 0).any(axis=0).sum()}")
else:
    print("\nNo TS_Quality matrix found")

In [ ]:
# Check what TS_DataMat values look like at special-quality positions
data_mat = sample_mat['TS_DataMat']
quality = sample_mat['TS_Quality']
special_mask = quality != 0

special_vals = data_mat[special_mask]
print(f"Special-quality positions: {special_mask.sum()}")
print(f"NaN at those positions: {np.isnan(special_vals).sum()}")
print(f"Inf at those positions: {np.isinf(special_vals).sum()}")
print(f"Finite at those positions: {np.isfinite(special_vals).sum()}")
print(f"\nUnique finite values at special positions (sample): {np.unique(special_vals[np.isfinite(special_vals)])[:20]}")

In [ ]:
df = all_tasks[all_tasks["dataset_name"] == "ElectricDevices"]

In [ ]:
df.dtypes

In [ ]:
invalid_dataset_count.sort_values(ascending=False)

In [ ]:

invalid_dataset_count = invalid_dataset_count[invalid_dataset_count > 0].sort_values(ascending=False)
n_datasets = filtered_tasks['dataset_name'].nunique()

print(f"Features with invalid values in ≥1 dataset: {len(invalid_dataset_count)} / {len(feature_cols)}")
print(f"Total datasets: {n_datasets}\n")
invalid_dataset_count.head(20)

In [ ]:
filtered_tasks.columns

In [ ]:
# Linear rescaling to unit interval [0, 1] per dataset, per feature
# As per paper: "For each dataset, each feature was linearly rescaled to the unit interval"
# Features with zero range (constant across all time series in a dataset) become NaN
feature_cols = [c for c in all_tasks.columns if c not in ('label', 'dataset_name')]

def normalize_features(group):
    feat = group[feature_cols]
    min_vals = feat.min()
    max_vals = feat.max()
    range_vals = max_vals - min_vals
    # Constant features (range=0) will become NaN via division by zero
    normalized = (feat - min_vals) / range_vals
    group[feature_cols] = normalized
    return group

all_tasks = all_tasks.groupby('dataset_name', group_keys=False).apply(normalize_features)
print(f"Shape: {all_tasks.shape}")
print(f"NaN count after normalization: {all_tasks[feature_cols].isna().sum().sum():,}")

In [ ]:
# Linear rescaling to unit interval [0, 1] per dataset, per feature
# As per paper: "For each dataset, each feature was linearly rescaled to the unit interval"
# Features with zero range (constant across all time series in a dataset) become NaN
feature_cols = [c for c in all_tasks.columns if c not in ('label', 'dataset_name')]

grouped = all_tasks.groupby('dataset_name')[feature_cols]
min_vals = grouped.transform('min')
max_vals = grouped.transform('max')
range_vals = max_vals - min_vals

# Constant features (range=0) will become NaN via division by zero
all_tasks[feature_cols] = (all_tasks[feature_cols] - min_vals) / range_vals

print(f"Shape: {all_tasks.shape}")
# print(f"NaN count after normalization: {all_tasks[feature_cols].isna().sum().sum():,}")

In [ ]:
all_tasks.head()

In [ ]:
# Count how many datasets have at least one invalid value per feature column
# Each dataset contributes at most 1 to the count per column
feature_cols = [c for c in all_tasks.columns if c not in ('label', 'dataset_name')]

def has_invalid(col):
    return col.isna().any() | np.isinf(col).any()

invalid_dataset_count = (
    all_tasks.groupby('dataset_name')[feature_cols]
    .apply(lambda grp: grp.apply(has_invalid))
    .sum()
)
invalid_dataset_count[invalid_dataset_count>0.8*len(task_names_UCR_2018)]

## old code

In [35]:
# -- Configure paths --
# Download .mat files from figshare link above into this directory
# Expected naming: HCTSA_{dataset_name}_N.mat
mat_dir = Path('../datasets/ucr_hctsa_features')

# Pick a dataset to test with (e.g., one of the UCR archive names)
dataset_name = 'Adiac'  # small dataset, good for testing
mat_path = mat_dir / f'HCTSA_{dataset_name}.mat'

print(f"Looking for: {mat_path}")
print(f"File exists: {mat_path.exists()}")

Looking for: ../datasets/ucr_hctsa_features/HCTSA_Adiac.mat
File exists: True


In [48]:
mat = sio.loadmat(mat_path)
ts = mat['TimeSeries']
print([str(ts[i,0]['filename'][0]) for i in range(5)])

ValueError: no field of name filename

In [54]:
mat['TS_DataMat'].shape

(781, 7658)

In [52]:
mat.keys()

dict_keys(['__header__', '__version__', '__globals__', 'MasterOperations', 'Operations', 'TS_DataMat', 'TS_Quality', 'TimeSeries', 'fromDatabase'])

In [49]:
print(ts.dtype.names)

('Name', 'Keywords', 'Length', 'Data', 'ID')


In [64]:
ts[0,0]['ID']

array([[1]], dtype=uint8)

In [50]:
# Check what Data looks like
print(ts[0,0]['Data'].shape)
print(ts[0,0]['ID'])

(176, 1)
[[1]]


In [70]:
from statsmodels.tsa.stattools import acf

def calc_co_f1ecac(x):
    """First lag where ACF drops below 1/e."""
    ac = acf(x, nlags=len(x)//2, fft=True)
    threshold = 1 / np.e
    below = np.where(ac < threshold)[0]
    return below[0] if len(below) > 0 else len(ac)

def calc_co_firstmin_ac(x):
    """First local minimum of ACF."""
    ac = acf(x, nlags=len(x)//2, fft=True)
    for i in range(1, len(ac) - 1):
        if ac[i] < ac[i - 1] and ac[i] <= ac[i + 1]:
            return i
    return len(ac)
f1ecac = calc_co_f1ecac(ts[0,0]['Data'])
calc_co_firstmin_ac(ts[0,0]['Data'])

41

In [ ]:
all_tasks

In [69]:
1/np.e

0.36787944117144233

In [66]:
f1ecac

np.int64(15)

In [68]:
import plotly.express as px
px.scatter(ts[0,0]['Data'
])

In [37]:
data, labels, operations = load_hctsa_mat(mat_path)


In [46]:
[n for n in operations['name'] if 'f1ecac' in n or 'FirstMin_ac' in n]

[]

In [39]:
df = pd.DataFrame.from_dict(operations)
df

,code_string,name,keywords,id,master_id
0,ST_length,length,"misc,raw,lengthdep",1,1
1,DN_mean,mean,"distribution,location,raw,locdep",2,5
2,DN_hmean,harmonic_mean,"distribution,location,raw,locdep",3,6
3,DN_median,median,"distribution,location,raw,locdep",4,8
4,DN_TrimmedMean_1,trimmed_mean_1,"distribution,location,raw,locdep",5,13
...,...,...,...,...,...
7653,MD_rawHRVmeas.tri10,MD_rawHRVmeas_tri10,"medical,raw",7654,1063
7654,MD_rawHRVmeas.tri20,MD_rawHRVmeas_tri20,"medical,raw",7655,1063
7655,MD_rawHRVmeas.trisqrt,MD_rawHRVmeas_trisqrt,"medical,raw",7656,1063
7656,MD_rawHRVmeas.SD1,MD_rawHRVmeas_SD1,"medical,raw,spreaddep",7657,1063


In [45]:
df[df['name'].str.contains("FirstMin")]

,code_string,name,keywords,id,master_id


In [ ]:

n_samples, n_features = data.shape
n_classes = len(np.unique(labels))
n_masked = data.mask.sum() if data.mask is not np.bool_(False) else 0

print(f"Dataset: {dataset_name}")
print(f"  {n_samples} time series samples (rows)")
print(f"  {n_features} hctsa features computed per sample (columns)")
print(f"  {n_classes} classes: {np.unique(labels)}")
print(f"  {n_masked} feature values are NaN/Inf ({n_masked/data.size:.1%} of matrix)")

In [ ]:
operations = pd.DataFrame.from_dict(operations)
operations

In [ ]:
operations[~operations['keywords'].str.contains('raw')]

In [ ]:
# What does the data look like as a classification problem?
# Each row = one time series, each column = one hctsa feature value
# Goal: use these features to predict the class label



print("Feature matrix (samples x features) — ready for classification:")
# print(df_features.head())
print(f"\nLabel distribution:")
print(df_features['label'].value_counts().sort_index())

In [ ]:
df_features.head()

## Build a simple DT for one task just as a test

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.model_selection import cross_validate                                                                                                                                                                                                                                               

X = data.data  # (781, 7658)
y = labels     # (781,)

# Replace any remaining NaN with column median for the classifier
X_clean = np.where(np.isnan(X), np.nanmedian(X, axis=0), X)

clf = DecisionTreeClassifier(random_state=42)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
                                                                                                                                                                                                                                                                                                   
results = cross_validate(clf, X_clean, y, cv=cv, return_estimator=True)                                                                                                                                                                                                                          
importances = np.array([est.feature_importances_ for est in results['estimator']])
                                                                                                                                                                                                                                                                                                   
# Now you have (10, 7658) — importance of each feature per fold                                                                                                                                                                                                                                  
mean_imp = importances.mean(axis=0)                                                                                                                                                                                                                                                              
std_imp = importances.std(axis=0)   

In [ ]:
feat_imp = pd.DataFrame(data=importances, columns=operations['name'])
mean_imp_dict = dict(sorted(zip(operations['name'], mean_imp), key=lambda x: x[1], reverse=True))

In [ ]:
top_10_features = list(mean_imp_dict.keys())[:10]
top_10_features

In [ ]:
px.box(feat_imp[top_10_features])

## Statistical pre-filtering